In [ ]:
%pip -q install yfinance scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
TICKER = "AAPL"       # Example: "RELIANCE.NS", "TCS.NS", "MSFT"
PERIOD = "5y"
INTERVAL = "1d"

raw_data = yf.download(
    TICKER,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)

if raw_data.empty:
    raise ValueError(
        f"No data returned for {TICKER}. Check the ticker symbol or try again."
    )

# yfinance may return MultiIndex columns depending on version/ticker.
if isinstance(raw_data.columns, pd.MultiIndex):
    raw_data.columns = raw_data.columns.get_level_values(0)

features = ["Open", "High", "Low", "Close", "Volume"]
missing = [col for col in features if col not in raw_data.columns]
if missing:
    raise KeyError(f"Expected columns missing from Yahoo Finance data: {missing}")

data = raw_data[features].dropna().copy()
data.index = pd.to_datetime(data.index)

print("Ticker:", TICKER)
print("Rows:", len(data))
print("Date range:", data.index.min().date(), "to", data.index.max().date())
display(data.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(data.index, data["Close"], label="Closing price")
axes[0].set_title(f"{TICKER} - Historical Closing Price")
axes[0].set_ylabel("Price")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(data.index, data["Volume"], label="Trading volume")
axes[1].set_title("Trading Volume")
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("Date")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
values = data[features].values.astype("float32")

# Chronological split: 70% train, 15% validation, 15% test.
n = len(values)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

if train_end <= 30 or val_end <= train_end or n - val_end < 2:
    raise ValueError("Not enough historical rows. Choose a longer period.")

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(values[:train_end])

scaled_values = scaler.transform(values)

close_column_index = features.index("Close")
close_min = scaler.data_min_[close_column_index]
close_max = scaler.data_max_[close_column_index]

print("Training rows:", train_end)
print("Validation rows:", val_end - train_end)
print("Test rows:", n - val_end)
print("Feature order:", features)
print("Scaled data shape:", scaled_values.shape)

In [ ]:
WINDOW_SIZE = 30

def make_windows(values_scaled, target_start, target_end, window_size=30):
    X, y, target_indices = [], [], []

    # target index is the day to predict; window ends on target_index - 1
    for target_idx in range(max(window_size, target_start), target_end):
        X.append(values_scaled[target_idx - window_size:target_idx, :])
        y.append(values_scaled[target_idx, close_column_index])
        target_indices.append(target_idx)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(target_indices, dtype=np.int64)
    )

X_train, y_train, train_indices = make_windows(
    scaled_values, WINDOW_SIZE, train_end, WINDOW_SIZE
)
X_val, y_val, val_indices = make_windows(
    scaled_values, train_end, val_end, WINDOW_SIZE
)
X_test, y_test, test_indices = make_windows(
    scaled_values, val_end, n, WINDOW_SIZE
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise ValueError("One split has no windows. Download more historical data.")

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW_SIZE, len(features))),
    tf.keras.layers.SimpleRNN(50, activation="tanh"),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)

model.summary()

In [ ]:
EPOCHS = 30
BATCH_SIZE = 32

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("RNN Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error (scaled Close)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Final training loss:", history.history["loss"][-1])
print("Final validation loss:", history.history["val_loss"][-1])

In [ ]:
pred_scaled = model.predict(X_test, verbose=0).reshape(-1)

def inverse_close(scaled_close_values):
    temp = np.zeros((len(scaled_close_values), len(features)), dtype=np.float32)
    temp[:, close_column_index] = scaled_close_values
    return scaler.inverse_transform(temp)[:, close_column_index]

y_test_actual = inverse_close(y_test)
pred_actual = inverse_close(pred_scaled)

mae = mean_absolute_error(y_test_actual, pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, pred_actual))
r2 = r2_score(y_test_actual, pred_actual)

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²:   {r2:.4f}")

In [ ]:
test_dates = data.index[test_indices]

plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test_actual, label="Actual Close", linewidth=1.8)
plt.plot(test_dates, pred_actual, label="Predicted Close", linewidth=1.5)
plt.title(f"{TICKER} - Actual vs RNN Predicted Closing Price")
plt.xlabel("Date")
plt.ylabel("Closing price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
latest_window = scaled_values[-WINDOW_SIZE:, :].reshape(
    1, WINDOW_SIZE, len(features)
)
next_close_scaled = model.predict(latest_window, verbose=0).reshape(-1)
next_close_estimate = inverse_close(next_close_scaled)[0]

print(f"Ticker: {TICKER}")
print("Latest available data date:", data.index[-1].date())
print(f"Estimated next-session Close: {next_close_estimate:.4f}")
print("This is an educational model estimate, not financial advice.")

In [ ]:
# The last input row's scaled Close is the previous trading day's Close.
baseline_scaled = X_test[:, -1, close_column_index]
baseline_actual = inverse_close(baseline_scaled)

baseline_mae = mean_absolute_error(y_test_actual, baseline_actual)
baseline_rmse = np.sqrt(mean_squared_error(y_test_actual, baseline_actual))

comparison = pd.DataFrame({
    "Model": ["RNN", "Persistence baseline"],
    "MAE": [mae, baseline_mae],
    "RMSE": [rmse, baseline_rmse]
})

display(comparison)